Load cortical and subcortical atlases, extract specific ROIs, and combine with new numbering (starting with 1, per [mrtrix3 requirements](https://mrtrix.readthedocs.io/en/latest/quantitative_structural_connectivity/structural_connectome.html#preparing-a-parcellation-image-for-connectome-generation)—see [labelconvert](https://mrtrix.readthedocs.io/en/latest/quantitative_structural_connectivity/labelconvert_tutorial.html#labelconvert-tutorial) for more details)

In [12]:
import os
import ants
import pandas as pd
import numpy as np
import nibabel as nib

from glob import glob
from nilearn.image import resample_to_img, binarize_img

In [18]:
ref_dir = os.path.abspath('/Users/dsj3886/data_local/reference')
print(ref_dir)

tian_fpath = os.path.join(ref_dir,'Tian_Subcortex_S3_7T.nii')
tian_img = nib.load(tian_fpath)
carpet_fpath = os.path.join(ref_dir,'tpl-MNI152NLin2009cAsym_space-MNI_res-01_label-carpet_atlas.nii')
carpet_img = nib.load(carpet_fpath)

/Users/dsj3886/data_local/reference


In [23]:
tian_dict = {
             'PUT-VA-lh': 42,
             'PUT-DA-lh': 43,
             'PUT-VP-lh': 44,
             'PUT-DP-lh': 45,
             'CAU-VA-lh': 46,
             'CAU-DA-lh': 47,
             'pCAU-lh': 54,
             'PUT-VA-rh': 15,
             'PUT-DA-rh': 16,
             'PUT-VP-rh': 17,
             'PUT-DP-rh': 18,
             'CAU-VA-rh': 19,
             'CAU-DA-rh': 20,
             'pCAU-rh': 27,
            }

carpet_dict = {'L-HG': 189, 
               'L-PP': 187, 'L-PT': 191, 
               'L-STGa': 117, 'L-STGp': 119,
               'R-HG': 190, 
               'R-PP': 188, 'R-PT': 192, 
               'R-STGa': 118, 'R-STGp': 120, }

In [24]:
def generate_mask(atlas_img, labelnum, labelname, new_labelnum=None):    
    atlas_data = atlas_img.get_fdata()
    atlas_affine = atlas_img.affine
    
    mask_data = np.zeros((atlas_data.shape))
    if new_labelnum:
        mask_data[np.where(atlas_data == labelnum)] = new_labelnum
    else:
        mask_data[np.where(atlas_data == labelnum)] = labelnum

    mask_img = nib.Nifti1Image(mask_data, atlas_affine)

    return mask_img

Extract ROIs

In [33]:
## COPY FROM FLT CODE
atlas_base = 'atlas-custom_subcort-tians3_cort-carpet'
out_dir = os.path.join('/Users/dsj3886/data_local/derivatives', atlas_base)
os.makedirs(out_dir, exist_ok=True)
print(out_dir)
new_lut = []
new_labelnum = 1

/Users/dsj3886/data_local/derivatives/atlas-custom_subcort-tians3_cort-carpet


In [34]:
# Tian atlas
for rx, region_label in enumerate(tian_dict.keys()):
    print(region_label)
    labelnum = tian_dict[region_label]

    mask_orig_img = generate_mask(tian_img, labelnum, region_label, new_labelnum)
    mask_img = image.resample_to_img(mask_orig_img, carpet_img, interpolation='nearest')

    out_base = f'{atlas_base}_{region_label}.nii.gz'
    out_fpath = os.path.join(out_dir, out_base)
    nib.save(mask_img, out_fpath)

    new_lut.append([new_labelnum, region_label])

    new_labelnum +=1

PUT-VA-lh
PUT-DA-lh
PUT-VP-lh
PUT-DP-lh
CAU-VA-lh
CAU-DA-lh
pCAU-lh
PUT-VA-rh
PUT-DA-rh
PUT-VP-rh
PUT-DP-rh
CAU-VA-rh
CAU-DA-rh
pCAU-rh


In [35]:
# Carpet aseg atlas
for rx, region_label in enumerate(carpet_dict.keys()):
    print(region_label)
    labelnum = carpet_dict[region_label]

    mask_img = generate_mask(carpet_img, labelnum, region_label, new_labelnum)

    out_base = f'{atlas_base}_{region_label}.nii.gz'
    out_fpath = os.path.join(out_dir, out_base)
    nib.save(mask_img, out_fpath)

    new_lut.append([new_labelnum, region_label])

    new_labelnum +=1


L-HG
L-PP
L-PT
L-STGa
L-STGp
R-HG
R-PP
R-PT
R-STGa
R-STGp


Combine ROIs into new atlas

In [36]:
# check new LUT
print(new_lut)

[[1, 'PUT-VA-lh'], [2, 'PUT-DA-lh'], [3, 'PUT-VP-lh'], [4, 'PUT-DP-lh'], [5, 'CAU-VA-lh'], [6, 'CAU-DA-lh'], [7, 'pCAU-lh'], [8, 'PUT-VA-rh'], [9, 'PUT-DA-rh'], [10, 'PUT-VP-rh'], [11, 'PUT-DP-rh'], [12, 'CAU-VA-rh'], [13, 'CAU-DA-rh'], [14, 'pCAU-rh'], [15, 'L-HG'], [16, 'L-PP'], [17, 'L-PT'], [18, 'L-STGa'], [19, 'L-STGp'], [20, 'R-HG'], [21, 'R-PP'], [22, 'R-PT'], [23, 'R-STGa'], [24, 'R-STGp']]


In [37]:
## COPY FROM FLT CODE
new_lut_df = pd.DataFrame(new_lut)
print(new_lut_df)

     0          1
0    1  PUT-VA-lh
1    2  PUT-DA-lh
2    3  PUT-VP-lh
3    4  PUT-DP-lh
4    5  CAU-VA-lh
5    6  CAU-DA-lh
6    7    pCAU-lh
7    8  PUT-VA-rh
8    9  PUT-DA-rh
9   10  PUT-VP-rh
10  11  PUT-DP-rh
11  12  CAU-VA-rh
12  13  CAU-DA-rh
13  14    pCAU-rh
14  15       L-HG
15  16       L-PP
16  17       L-PT
17  18     L-STGa
18  19     L-STGp
19  20       R-HG
20  21       R-PP
21  22       R-PT
22  23     R-STGa
23  24     R-STGp


In [38]:
# save new LUT
new_lut_fpath = os.path.join(out_dir, atlas_base+'_lut.tsv')

new_lut_df.to_csv(new_lut_fpath, sep='\t', header=False, index=False)

In [76]:
# save new atlas
from nilearn import image

atlas_4d = image.load_img(out_dir+'/{}*.gz'.format(atlas_base), wildcards=True)
atlas_float = image.math_img('np.max(img, axis=-1)', img=atlas_4d) # np.sum

# convert data to integers and create new nib img
atlas_int_data = nib.casting.float_to_int(atlas.get_fdata(), np.int8)
atlas_img = image.new_img_like(atlas_float, atlas_int_data)
atlas_img.set_data_dtype(np.int8)

atlas_joined_fpath = os.path.join(out_dir, atlas_base+'_atlas.nii.gz')
nib.save(atlas_img, atlas_joined_fpath)

In [89]:
atlas_joined_fpath

'/Users/dsj3886/data_local/derivatives/atlas-custom_subcort-tians3_cort-carpet/atlas-custom_subcort-tians3_cort-carpet_atlas.nii.gz'

## warp atlas to MNI space used by HCP

In [103]:
from ants import apply_transforms

In [113]:
xfm_fpath = '/Users/dsj3886/data_local/reference/tpl-MNI152NLin2009cAsym_from-MNI152NLin6Asym_mode-image_xfm.h5'
fixed_fpath = '/Users/dsj3886/data_local/HCP_3T_structural/100610/MNINonLinear/T1w.nii.gz'
moving_fpath = atlas_joined_fpath
invert_list = [True]

x_img = apply_transforms(fixed=fixed_fpath, 
                 moving=atlas_joined_fpath, 
                 transformlist=[xfm_fpath], 
                 interpolator='genericLabel',
                 #imagetype=0,
                 verbose=True,
                 )


In [112]:
print(x_img)

None


In [98]:
help(apply_transforms)

Help on function apply_transforms in module ants.registration.apply_transforms:

apply_transforms(fixed, moving, transformlist, interpolator='linear', imagetype=0, whichtoinvert=None, compose=None, defaultvalue=0, verbose=False, **kwargs)
    Apply a transform list to map an image from one domain to another.
    In image registration, one computes mappings between (usually) pairs
    of images. These transforms are often a sequence of increasingly
    complex maps, e.g. from translation, to rigid, to affine to deformation.
    The list of such transforms is passed to this function to interpolate one
    image domain into the next image domain, as below. The order matters
    strongly and the user is advised to familiarize with the standards
    established in examples.

    ANTsR function: `antsApplyTransforms`

    Arguments
    ---------
    fixed : ANTsImage
        fixed image defining domain into which the moving image is transformed.

    moving : AntsImage
        moving image t